# Extension Step 2 — Task Verification Baseline

Trains a Transformer-based video-level binary classifier using Leave-One-Out (LOO) cross-validation.
For each recipe k, the model is trained on all other recipes and tested on recipe k.

**Prerequisites:**
- Step embeddings in `STEP_EMBEDDINGS_DIR` (output of Extension Step 1)

**Output:**
- Checkpoints in `STEP2_RESULTS_DIR/checkpoints/fold_*_best.pt`
- Metrics in `STEP2_RESULTS_DIR/results.csv`

In [1]:
# ── 1. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [2]:
# ── 2. Path constants (FIXED — do not modify) ─────────────────────────────────
DRIVE_ROOT           = '/content/drive/MyDrive/AML_Project'
STEP_EMBEDDINGS_DIR  = f'{DRIVE_ROOT}/step1/step_embeddings'
STEP2_RESULTS_DIR    = f'{DRIVE_ROOT}/step2/results'
REPO_DIR             = '/content/code'

# annotations/annotation_json/complete_step_annotations.json
# is populated by git clone --recursive (annotations submodule)
ANNOTATIONS_PATH     = f'{REPO_DIR}/annotations/annotation_json/complete_step_annotations.json'

# local copy of step embeddings (faster I/O than Drive during training)
LOCAL_EMBEDDINGS_DIR = '/content/step_embeddings'

print('Paths defined.')

Paths defined.


In [3]:
# ── 3. Clone repo (--recursive fetches annotations submodule) ─────────────────
!git clone --recursive https://github.com/Laio95/aml-2025-mistake-detection.git {REPO_DIR}

Cloning into '/content/code'...
remote: Enumerating objects: 646, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 646 (delta 164), reused 179 (delta 137), pack-reused 415 (from 2)
Receiving objects: 100% (646/646), 5.03 MiB | 8.46 MiB/s, done.
Resolving deltas: 100% (397/397), done.
Submodule 'actionformer_release' (https://github.com/rohithpeddi/actionformer_release.git) registered for path 'actionformer_release'
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/actionformer_release'...
remote: Enumerating objects: 410, done.        
remote: Counting objects: 100% (26/26), done.        
remote: Compressing objects: 100% (20/20), done.        
remote: Total 410 (delta 16), reused 6 (delta 6), pack-reused 384 (from 2)        
Receiving objects: 100% (410/410), 651.07 KiB | 28.31 MiB/s, done.
Resolving deltas: 100% (225/225), done.
Clonin

In [4]:
# ── 4. Install dependencies ───────────────────────────────────────────────────
!pip install -r {REPO_DIR}/requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.


In [5]:
# ── 5. Copy step embeddings to local storage (faster than reading from Drive) ──
# 384 small .npz files — copy once, reused across all LOO folds and epochs
import os
os.makedirs(LOCAL_EMBEDDINGS_DIR, exist_ok=True)
!cp -r {STEP_EMBEDDINGS_DIR}/* {LOCAL_EMBEDDINGS_DIR}/
print(f"Copied: {len(os.listdir(LOCAL_EMBEDDINGS_DIR))} files → {LOCAL_EMBEDDINGS_DIR}")

Copied: 384 files → /content/step_embeddings


In [6]:
# ── 6. WandB login ────────────────────────────────────────────────────────────
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: albertogiunti2001 (albertogiunti2001-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Full LOO training — 50 epochs

Allena il `TaskVerifier` su tutti i fold LOO (uno per ricetta).  
I checkpoint migliori vengono salvati in `STEP2_RESULTS_DIR/checkpoints/`.  
Le metriche finali (mean ± std di AUC, F1, Accuracy) vengono salvate in `STEP2_RESULTS_DIR/results.csv`.

**Tempo stimato:** ~3-5 min per fold × 24 fold ≈ 1.5-2 ore su T4.

In [ ]:
%%bash -s "$REPO_DIR" "$ANNOTATIONS_PATH" "$LOCAL_EMBEDDINGS_DIR" "$STEP2_RESULTS_DIR"
cd $1

python -m extension.step2.loo_train \
    --annotations_path    "$2" \
    --step_embeddings_dir "$3" \
    --output_dir          "$4" \
    --num_epochs          50 \
    --lr                  1e-4 \
    --weight_decay        1e-3 \
    --num_layers          2 \
    --dropout             0.1 \
    --threshold           0.5 \
    --seed                42 \
    --num_workers         2 \
    --enable_wandb

Device: cuda
Loaded 384 samples | correct=164, incorrect=220 | recipes=24 | skipped=0 (no .npz)
Class distribution — correct: 164, incorrect: 220
pos_weight = 0.7455

LOO: 24 folds (one per recipe)


Fold 1/24 — Recipe 1 | train=366  test=18
  Fold 1 | Recipe  1 | Epoch   1 | train_loss=1.1526  test_loss=0.6641  AUC=0.6769  F1=0.8387  Acc=0.7222  lr=1.00e-04
  Fold 1 | Recipe  1 | Epoch   2 | train_loss=0.8822  test_loss=1.3236  AUC=0.7077  F1=0.8387  Acc=0.7222  lr=1.00e-04
  Fold 1 | Recipe  1 | Epoch   3 | train_loss=0.6941  test_loss=1.4353  AUC=0.7077  F1=0.8387  Acc=0.7222  lr=1.00e-04
  Fold 1 | Recipe  1 | Epoch   4 | train_loss=0.5754  test_loss=0.9827  AUC=0.8000  F1=0.8387  Acc=0.7222  lr=1.00e-04
  Fold 1 | Recipe  1 | Epoch   5 | train_loss=0.3609  test_loss=1.9020  AUC=0.7538  F1=0.8387  Acc=0.7222  lr=1.00e-04
  Fold 1 | Recipe  1 | Epoch   6 | train_loss=0.3159  test_loss=1.4657  AUC=0.7692  F1=0.8387  Acc=0.7222  lr=1.00e-04
  Fold 1 | Recipe  1 | Epoch   7 | train_los

## Risultati — verifica e summary

In [ ]:
# ── Verifica checkpoint salvati ───────────────────────────────────────────────
import pathlib
ckpt_dir = pathlib.Path(STEP2_RESULTS_DIR) / 'checkpoints'
ckpts = sorted(ckpt_dir.glob('*.pt'))
print(f"{len(ckpts)} checkpoint(s) salvati:")
for p in ckpts:
    print(f"  {p.name}")

24 checkpoint(s) salvati:
  fold_10_recipe_12_best.pt
  fold_11_recipe_13_best.pt
  fold_12_recipe_15_best.pt
  fold_13_recipe_16_best.pt
  fold_14_recipe_17_best.pt
  fold_15_recipe_18_best.pt
  fold_16_recipe_20_best.pt
  fold_17_recipe_21_best.pt
  fold_18_recipe_22_best.pt
  fold_19_recipe_23_best.pt
  fold_1_recipe_1_best.pt
  fold_20_recipe_25_best.pt
  fold_21_recipe_26_best.pt
  fold_22_recipe_27_best.pt
  fold_23_recipe_28_best.pt
  fold_24_recipe_29_best.pt
  fold_2_recipe_2_best.pt
  fold_3_recipe_3_best.pt
  fold_4_recipe_4_best.pt
  fold_5_recipe_5_best.pt
  fold_6_recipe_7_best.pt
  fold_7_recipe_8_best.pt
  fold_8_recipe_9_best.pt
  fold_9_recipe_10_best.pt


In [ ]:
# ── Leggi e stampa results.csv ────────────────────────────────────────────────
import pandas as pd

csv_path = f'{STEP2_RESULTS_DIR}/results.csv'
df = pd.read_csv(csv_path)
print(df.to_string(index=False))

    fold activity_id                auc                 f1            accuracy
       1           1 0.9076923076923078 0.8666666666666667  0.7777777777777778
       2           2 0.9166666666666667               0.75                0.75
       3           3                0.7 0.2222222222222222 0.46153846153846156
       4           4 0.8571428571428572 0.7368421052631579  0.7058823529411765
       5           5 0.7142857142857143 0.6363636363636364  0.4666666666666667
       6           7                0.7 0.7272727272727273               0.625
       7           8                0.8 0.7692307692307693               0.625
       8           9 0.7111111111111111 0.6153846153846154  0.6428571428571429
       9          10            0.78125 0.8235294117647058                0.75
      10          12 0.8051948051948052 0.6086956521739131                 0.5
      11          13                1.0                0.8  0.7857142857142857
      12          15                0.7 0.8181818181